In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist         # библиотека базы выборок Mnist
from tensorflow import keras
from tensorflow.keras.layers import Dense, Flatten, Dropout, BatchNormalization

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

limit = 5000
x_train_data = x_train[:limit]
y_train_data = y_train_cat[:limit]

x_valid = x_train[limit:limit*2]
y_valid = y_train_cat[limit:limit*2]

model = keras.Sequential([
    Flatten(input_shape=(28, 28, 1)),
    Dense(300, activation='relu'),
    # Dropout(0.8),
    BatchNormalization(),
    Dense(10, activation='softmax') ])

# print(model.summary())      # вывод структуры НС в консоль

model.compile(optimizer='adam',
             loss='categorical_crossentropy',
             metrics=['accuracy'])


his = model.fit(x_train_data, y_train_data, epochs=50, batch_size=32, validation_data=(x_valid, y_valid))


plt.plot(his.history['loss'])
plt.plot(his.history['val_loss'])
plt.show()

## Izah

## BatchNormalization

In [ ]:
model = keras.Sequential([
    Flatten(input_shape=(28, 28, 1)),
    Dense(300, activation='relu'),
    BatchNormalization(),
    # BatchNormalization — hər qatın (layer) çıxışını
    # növbəti qata ötürmədən əvvəl NORMALLAŞDIRAN
    # (ortalamanı 0, dispersiyanı 1-ə yaxınlaşdıran)
    # bir texnikadır.
    #
    # Niyə lazımdır?
    # → Öyrətmə zamanı əvvəlki qatların çəkiləri (weights)
    #    daim dəyişdiyi üçün, hər qata gələn datanın
    #    paylanması (distribution) da daim dəyişir.
    #    Bu problem "internal covariate shift" adlanır.
    #
    # → BatchNormalization hər mini-batch üçün gələn
    #    dataları normallaşdıraraq bu "sürüşməni" azaldır,
    #    beləliklə növbəti qat daha sabit/proqnozlaşdırıla
    #    bilən bir giriş alır.
    #
    # Necə işləyir? (hər mini-batch üçün, hər feature üzrə)
    #
    # 1. Batch-in ortası (μ) və dispersiyası (σ²) hesablanır
    # 2. Dəyərlər normallaşdırılır:
    #        x̂ = (x - μ) / √(σ² + ε)
    # 3. Sonra iki öyrənilə bilən parametrlə (γ, β)
    #    yenidən miqyaslandırılır:
    #        y = γ·x̂ + β
    #    (bu, modelin lazım gələrsə normallaşdırmanı
    #     qismən "geri götürməsinə" imkan verir)
    #
    # Faydaları:
    # → Öyrətmə xeyli SÜRƏTLƏNİR (daha yüksək learning
    #    rate istifadə etməyə imkan verir)
    # → Modelin çəkilərin başlanğıc dəyərinə (initialization)
    #    həssaslığını azaldır
    # → Yüngül regularization təsiri də var (Dropout-a
    #    bənzər, çünki hər batch fərqli statistika verir) —
    #    ona görə bəzən Dropout-un yerinə/əlavə istifadə olunur
    # → Gradient partlaması/yoxa çıxması (exploding/vanishing
    #    gradient) problemlərini azaldır
    #
    # Harada yerləşdirilir?
    # → Adətən Dense (və ya Conv) qatından SONRA,
    #    aktivasiya funksiyasından (activation) ƏVVƏL və ya
    #    SONRA istifadə olunur (hər iki yanaşma da mövcuddur,
    #    praktikada nəticələr fərqli ola bilər)
    #
    # QEYD: Dropout-dan fərqli olaraq, BatchNormalization
    # həm TƏLİM, həm də TEST zamanı fəaldır — sadəcə test
    # zamanı batch-in real ortası/dispersiyası əvəzinə,
    # təlim boyu toplanmış "hərəkətli ortalama" (moving
    # average) istifadə olunur.
    #
    # QEYD 2: BatchNormalization ilə Dropout eyni modeldə
    # birlikdə istifadə edildikdə diqqətli olmaq lazımdır —
    # bəzən ikisi bir yerdə gözlənilməz nəticələr verə bilər,
    # ona görə çoxları ya yalnız BatchNorm, ya da yalnız
    # Dropout istifadə etməyi tövsiyə edir.

    Dense(10, activation='softmax')
])